## Date Drift Amortization Audit Summary

**1. Task Process:**

A Python script was used to perform a precise amortization audit, leveraging `datetime` and `decimal` libraries. The process involved:

*   **Initialization:** Defining loan principal ($100,000), annual rate (12%/365 days), and start date (Jan 1, 2024).
*   **Payment Processing:** Chronologically applying irregular payments (Jan 31: $5,000, Feb 29: $5,000, Mar 31: $5,000).
*   **Daily Interest Calculation:** Accurately computing `days_elapsed` between payments using `datetime.date`, correctly accounting for the 2024 leap year and February 29th. Interest was calculated daily as `Balance * (Annual Rate / 365) * Days_Elapsed`.
*   **Payment Application:** Applying payments first to accrued interest, then to principal.
*   **Tracking:** Maintaining updated principal balance and total interest charged.
*   **Output:** Generating a detailed, period-by-period amortization breakdown.

**2. Results and Evaluation of the Task:**

The audit successfully performed a precise amortization calculation. It accurately accounted for irregular payment dates and the 2024 leap year, confirming robust calendar math. The total interest charged across all periods was **$2838.10**, resulting in an ending balance of **$87838.10** after the last payment.

Prompt:
The Scenario: You are auditing a small business loan that has irregular payment dates. The borrower claims the bank overcharged them interest.
Loan Principal: $100,000
Annual Rate: 12% (calculated daily: Balance * (0.12 / 365) * Days_Elapsed)
Start Date: Jan 1, 2024 (Leap Year)
Payment Log:
Jan 31: 5,000
Feb 29: 5,000 (Leap Day)
Mar 31: 5,000

Here are the days per month
January:31
February:29
March:31
April:30
May:31
June:30
July:31
August:31
September:30
October:31
November:30
December:31

In [ ]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP

# -----------------------------
# Inputs (edit these if needed)
# -----------------------------
principal = Decimal("100000.00")
annual_rate = Decimal("0.12")
day_count_base = Decimal("365")  # per your rule: 0.12/365 (even though 2024 is leap year)

start_date = date(2024, 1, 1)

# Payment log: (payment_date, payment_amount)
payments = [
    (date(2024, 1, 31), Decimal("5000.00")),
    (date(2024, 2, 29), Decimal("5000.00")),
    (date(2024, 3, 31), Decimal("5000.00")),
]

# -----------------------------
# Helpers
# -----------------------------
def q2(x: Decimal) -> Decimal:
    """Round to 2dp like a bank statement."""
    return x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

daily_rate = annual_rate / day_count_base

# -----------------------------
# Audit calculation
# Assumptions:
# - Interest accrues up to (but not including) the payment date
# - Payment applies to interest first, then principal
# - Simple interest (no compounding)
# -----------------------------
balance = principal
total_interest = Decimal("0.00")

rows = []
prev_date = start_date

for pay_date, pay_amt in payments:
    days_elapsed = (pay_date - prev_date).days  # accrues from prev_date to pay_date (excluding pay_date)
    interest = balance * daily_rate * Decimal(days_elapsed)

    # Apply payment: interest first, remainder to principal
    interest_paid = min(pay_amt, interest)
    principal_paid = pay_amt - interest_paid
    new_balance = balance - principal_paid

    total_interest += interest

    rows.append({
        "period_start": prev_date,
        "period_end": pay_date,
        "days": days_elapsed,
        "start_balance": q2(balance),
        "interest": q2(interest),
        "payment": q2(pay_amt),
        "interest_paid": q2(interest_paid),
        "principal_paid": q2(principal_paid),
        "end_balance": q2(new_balance),
    })

    # Move to next period
    balance = new_balance
    prev_date = pay_date

# -----------------------------
# Print results
# -----------------------------
print(f"Daily rate = {daily_rate}  (annual_rate/day_count_base = {annual_rate}/{day_count_base})\n")

for i, r in enumerate(rows, 1):
    print(f"Period {i}: {r['period_start']} -> {r['period_end']}  ({r['days']} days)")
    print(f"  Start balance : ${r['start_balance']}")
    print(f"  Interest      : ${r['interest']}")
    print(f"  Payment       : ${r['payment']}")
    print(f"    Interest paid  : ${r['interest_paid']}")
    print(f"    Principal paid : ${r['principal_paid']}")
    print(f"  End balance   : ${r['end_balance']}\n")

print(f"Total interest charged across all periods: ${q2(total_interest)}")
print(f"Ending balance after last payment:         ${q2(balance)}")


Daily rate = 0.0003287671232876712328767123288  (annual_rate/day_count_base = 0.12/365)

Period 1: 2024-01-01 -> 2024-01-31  (30 days)
  Start balance : $100000.00
  Interest      : $986.30
  Payment       : $5000.00
    Interest paid  : $986.30
    Principal paid : $4013.70
  End balance   : $95986.30

Period 2: 2024-01-31 -> 2024-02-29  (29 days)
  Start balance : $95986.30
  Interest      : $915.16
  Payment       : $5000.00
    Interest paid  : $915.16
    Principal paid : $4084.84
  End balance   : $91901.46

Period 3: 2024-02-29 -> 2024-03-31  (31 days)
  Start balance : $91901.46
  Interest      : $936.64
  Payment       : $5000.00
    Interest paid  : $936.64
    Principal paid : $4063.36
  End balance   : $87838.10

Total interest charged across all periods: $2838.10
Ending balance after last payment:         $87838.10
